# NB_02 — Silver Clinical Transformation

## CMS-HCC V22 → V28 Medicare Risk Transition Platform

### Objective

This notebook transforms validated CMS DE-SynPUF Bronze data into
analytics-ready Silver datasets for V22 → V28 HCC transition analysis.

### Major Transformations

1. Standardize beneficiary records
2. Standardize inpatient and outpatient claims
3. Convert service dates and monetary fields
4. Normalize ICD-9-CM diagnoses
5. Apply CMS GEM ICD-9 → ICD-10 mappings
6. Build diagnosis-level claim records
7. Apply CMS-HCC V22 ICD-10 mappings
8. Apply CMS-HCC V28 ICD-10 mappings
9. Derive member-level HCC assignments
10. Create Silver member, claim, diagnosis, and HCC tables

### Design Principle

Bronze preserves raw CMS source data.

Silver performs clinical and analytical standardization while retaining
traceability back to the original claim and beneficiary identifiers.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import setuptools
import pkg_resources
import hccpy

from hccpy.hcc import HCCEngine

print("setuptools:", setuptools.__version__)
print("pkg_resources loaded")
print("hccpy loaded")


In [ ]:
df_bene_2008 = spark.table(
    "Bronze_HCC.dbo.bronze_beneficiary_2008"
)

df_bene_2009 = spark.table(
    "Bronze_HCC.dbo.bronze_beneficiary_2009"
)

df_bene_2010 = spark.table(
    "Bronze_HCC.dbo.bronze_beneficiary_2010"
)

df_inpatient = spark.table(
    "Bronze_HCC.dbo.bronze_inpatient_claims"
)

df_outpatient = spark.table(
    "Bronze_HCC.dbo.bronze_outpatient_claims"
)

df_gem_raw = spark.table(
    "Bronze_HCC.dbo.bronze_icd9_icd10_gem_raw"
)

In [ ]:
print("2008 beneficiaries:", df_bene_2008.count())
print("2009 beneficiaries:", df_bene_2009.count())
print("2010 beneficiaries:", df_bene_2010.count())

print("Inpatient claims:", df_inpatient.count())
print("Outpatient claims:", df_outpatient.count())
print("GEM rows:", df_gem_raw.count())

## 1. ICD-9-CM → ICD-10-CM GEM Crosswalk

CMS DE-SynPUF claims contain ICD-9-CM diagnosis codes, while the HCC V22
and V28 model mappings operate on ICD-10-CM diagnoses.

The CMS 2018 General Equivalence Mapping (GEM) is parsed into a structured
Silver reference table.

All mappings are retained for traceability. A deterministic canonical
mapping is also derived for analytical claim conversion by prioritizing:

1. mapped records,
2. exact mappings over approximate mappings,
3. non-combination mappings,
4. deterministic scenario and choice-list ordering.

The resulting crosswalk is used for retrospective analytical simulation,
not clinical recoding.

In [ ]:
# Parse the raw GEM table
df_gem = (
    df_gem_raw
    .select(
        F.trim(
            F.substring("value", 1, 5)
        ).alias("icd9_code"),

        F.trim(
            F.substring("value", 7, 7)
        ).alias("icd10_code"),

        F.substring("value", 15, 1)
        .cast("int")
        .alias("approximate_flag"),

        F.substring("value", 16, 1)
        .cast("int")
        .alias("no_map_flag"),

        F.substring("value", 17, 1)
        .cast("int")
        .alias("combination_flag"),

        F.substring("value", 18, 1)
        .cast("int")
        .alias("scenario"),

        F.substring("value", 19, 1)
        .cast("int")
        .alias("choice_list")
    )
    .filter(
        F.col("icd9_code") != ""
    )
)

In [ ]:
# Normalize the codes
df_gem = (
    df_gem
    .withColumn(
        "icd9_code_normalized",
        F.upper(
            F.regexp_replace(
                F.trim("icd9_code"),
                "\\.",
                ""
            )
        )
    )
    .withColumn(
        "icd10_code_normalized",
        F.upper(
            F.regexp_replace(
                F.trim("icd10_code"),
                "\\.",
                ""
            )
        )
    )
)

In [ ]:
gem_summary = (
    df_gem
    .agg(
        F.count("*").alias("mapping_rows"),

        F.countDistinct(
            "icd9_code_normalized"
        ).alias("distinct_icd9_codes"),

        F.countDistinct(
            "icd10_code_normalized"
        ).alias("distinct_icd10_codes"),

        F.sum(
            F.when(
                F.col("no_map_flag") == 1,
                1
            ).otherwise(0)
        ).alias("no_map_rows"),

        F.sum(
            F.when(
                F.col("approximate_flag") == 1,
                1
            ).otherwise(0)
        ).alias("approximate_rows"),

        F.sum(
            F.when(
                F.col("combination_flag") == 1,
                1
            ).otherwise(0)
        ).alias("combination_rows")
    )
)

display(gem_summary)

In [ ]:
gem_quality = (
    df_gem
    .groupBy(
        "approximate_flag",
        "combination_flag"
    )
    .count()
    .orderBy(
        "approximate_flag",
        "combination_flag"
    )
)

display(gem_quality)

In [ ]:
gem_quality_pd = (
    gem_quality
    .withColumn(
        "mapping_type",
        F.concat(
            F.lit("Approx="),
            F.col("approximate_flag"),
            F.lit(", Comb="),
            F.col("combination_flag")
        )
    )
    .select(
        "mapping_type",
        "count"
    )
    .toPandas()
)

plt.figure(figsize=(8, 5))

plt.bar(
    gem_quality_pd["mapping_type"],
    gem_quality_pd["count"]
)

plt.title(
    "CMS GEM ICD-9 → ICD-10 Mapping Quality"
)

plt.xlabel("Mapping category")
plt.ylabel("Number of mappings")

plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# canonical GEM mapping
mapping_window = (
    Window
    .partitionBy("icd9_code_normalized")
    .orderBy(
        F.col("approximate_flag").asc(),
        F.col("combination_flag").asc(),
        F.col("scenario").asc(),
        F.col("choice_list").asc(),
        F.col("icd10_code_normalized").asc()
    )
)

df_gem_canonical = (
    df_gem
    .filter(F.col("no_map_flag") == 0)
    .withColumn(
        "mapping_rank",
        F.row_number().over(mapping_window)
    )
    .filter(F.col("mapping_rank") == 1)
    .drop("mapping_rank")
)

In [ ]:
canonical_rows = df_gem_canonical.count()

canonical_distinct = (
    df_gem_canonical
    .select("icd9_code_normalized")
    .distinct()
    .count()
)

print("Canonical rows:", canonical_rows)
print("Distinct ICD-9 codes:", canonical_distinct)
print(
    "Unique mapping:",
    canonical_rows == canonical_distinct
)

In [ ]:
(
    df_gem
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_icd9_icd10_gem")
)

(
    df_gem_canonical
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_icd9_icd10_canonical")
)

## 2. Standardized Silver Claims

CMS DE-SynPUF inpatient and outpatient claims are standardized into a
single analytical claim structure.

The Silver claim layer:

- creates a globally unique claim identifier,
- standardizes member identifiers,
- parses service dates,
- preserves source-specific claim type,
- standardizes payment fields,
- derives inpatient length of stay,
- retains all ten ICD-9 diagnosis positions,
- and preserves source lineage.

No ICD-10 conversion or HCC assignment is performed yet in this step.

In [ ]:
diagnosis_cols = [
    f"ICD9_DGNS_CD_{i}"
    for i in range(1, 11)
]

# Standardize inpatient claims
df_inpatient_std = (
    df_inpatient

    .withColumn(
        "claim_id",
        F.concat(
            F.lit("IP_"),
            F.col("CLM_ID").cast("string"),
            F.lit("_"),
            F.col("SEGMENT").cast("string")
        )
    )

    .withColumn(
        "member_id",
        F.col("DESYNPUF_ID").cast("string")
    )

    .withColumn(
        "source_system",
        F.lit("CMS_DESYNPUF")
    )

    .withColumn(
        "claim_type",
        F.lit("INPATIENT")
    )

    .withColumn(
        "service_date",
        F.to_date(
            F.col("CLM_FROM_DT").cast("string"),
            "yyyyMMdd"
        )
    )

    .withColumn(
        "service_date_end",
        F.to_date(
            F.col("CLM_THRU_DT").cast("string"),
            "yyyyMMdd"
        )
    )

    .withColumn(
        "paid_amount",
        F.col("CLM_PMT_AMT").cast("double")
    )

    .withColumn(
        "primary_payer_amount",
        F.coalesce(
            F.col("NCH_PRMRY_PYR_CLM_PD_AMT"),
            F.lit(0.0)
        )
    )

    .withColumn(
        "los_days",
        F.coalesce(
            F.col("CLM_UTLZTN_DAY_CNT"),
            F.datediff(
                F.col("service_date_end"),
                F.col("service_date")
            )
        ).cast("int")
    )

    .withColumn(
        "provider_npi",
        F.col("AT_PHYSN_NPI").cast("string")
    )

    .withColumn(
        "icd9_primary",
        F.col("ICD9_DGNS_CD_1")
    )

    .withColumn(
        "icd9_array",
        F.array(
            *[F.col(c) for c in diagnosis_cols]
        )
    )

    .withColumn(
        "icd9_array",
        F.expr(
            "filter(icd9_array, x -> x is not null)"
        )
    )
)

# Standardize outpatient claims
df_outpatient_std = (
    df_outpatient

    .withColumn(
        "claim_id",
        F.concat(
            F.lit("OP_"),
            F.col("CLM_ID").cast("string"),
            F.lit("_"),
            F.col("SEGMENT").cast("string")
        )
    )

    .withColumn(
        "member_id",
        F.col("DESYNPUF_ID").cast("string")
    )

    .withColumn(
        "source_system",
        F.lit("CMS_DESYNPUF")
    )

    .withColumn(
        "claim_type",
        F.lit("OUTPATIENT")
    )

    .withColumn(
        "service_date",
        F.to_date(
            F.col("CLM_FROM_DT").cast("string"),
            "yyyyMMdd"
        )
    )

    .withColumn(
        "service_date_end",
        F.to_date(
            F.col("CLM_THRU_DT").cast("string"),
            "yyyyMMdd"
        )
    )

    .withColumn(
        "paid_amount",
        F.col("CLM_PMT_AMT").cast("double")
    )

    .withColumn(
        "primary_payer_amount",
        F.coalesce(
            F.col("NCH_PRMRY_PYR_CLM_PD_AMT"),
            F.lit(0.0)
        )
    )

    .withColumn(
        "los_days",
        F.lit(0).cast("int")
    )

    .withColumn(
        "provider_npi",
        F.col("AT_PHYSN_NPI").cast("string")
    )

    .withColumn(
        "icd9_primary",
        F.col("ICD9_DGNS_CD_1")
    )

    .withColumn(
        "icd9_array",
        F.array(
            *[F.col(c) for c in diagnosis_cols]
        )
    )

    .withColumn(
        "icd9_array",
        F.expr(
            "filter(icd9_array, x -> x is not null)"
        )
    )
)

# Select common Silver schema
claim_columns = [
    "claim_id",
    "member_id",
    "source_system",
    "claim_type",
    "service_date",
    "service_date_end",
    "paid_amount",
    "primary_payer_amount",
    "los_days",
    "provider_npi",
    "icd9_primary",
    "icd9_array"
]

df_inpatient_std = (
    df_inpatient_std
    .select(*claim_columns)
)

df_outpatient_std = (
    df_outpatient_std
    .select(*claim_columns)
)

df_silver_claim = (
    df_inpatient_std
    .unionByName(df_outpatient_std)
)

In [ ]:
# Validate 
claim_validation = (
    df_silver_claim
    .agg(
        F.count("*")
        .alias("total_claims"),

        F.countDistinct("claim_id")
        .alias("distinct_claim_ids"),

        F.countDistinct("member_id")
        .alias("distinct_members"),

        F.sum(
            F.when(
                F.col("claim_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_claim_ids"),

        F.sum(
            F.when(
                F.col("member_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_member_ids"),

        F.sum(
            F.when(
                F.col("service_date").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_service_dates")
    )
)

display(claim_validation)

In [ ]:
# Claim type distribution
claim_type_counts = (
    df_silver_claim
    .groupBy("claim_type")
    .count()
    .orderBy(F.desc("count"))
)

display(claim_type_counts)

In [ ]:
# Plot Claim Type Distribution
import matplotlib.pyplot as plt

claim_type_pd = claim_type_counts.toPandas()

plt.figure(figsize=(7, 5))

plt.bar(
    claim_type_pd["claim_type"],
    claim_type_pd["count"]
)

plt.title("CMS DE-SynPUF Claim Type Distribution")
plt.xlabel("Claim type")
plt.ylabel("Number of claims")

plt.tight_layout()
plt.show()

In [ ]:
df_silver_claim = (
    df_silver_claim
    .withColumn(
        "diagnosis_count",
        F.size("icd9_array")
    )
)

display(
    df_silver_claim
    .groupBy(
        "claim_type"
    )
    .agg(
        F.round(
            F.avg("diagnosis_count"),
            2
        ).alias("avg_diagnoses"),

        F.max(
            "diagnosis_count"
        ).alias("max_diagnoses")
    )
)

In [ ]:
# Plot Diagnosis count distribution
diagnosis_distribution = (
    df_silver_claim
    .groupBy(
        "claim_type",
        "diagnosis_count"
    )
    .count()
    .orderBy(
        "claim_type",
        "diagnosis_count"
    )
)

diagnosis_pd = diagnosis_distribution.toPandas()

plt.figure(figsize=(9, 5))

for claim_type in diagnosis_pd["claim_type"].unique():
    subset = diagnosis_pd[
        diagnosis_pd["claim_type"] == claim_type
    ]

    plt.plot(
        subset["diagnosis_count"],
        subset["count"],
        marker="o",
        label=claim_type
    )

plt.title("Diagnosis Count Distribution by Claim Type")
plt.xlabel("Diagnoses per claim")
plt.ylabel("Number of claims")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
(
    df_silver_claim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_claim")
)
print(
    "Silver claims:",
    spark.table("silver_claim").count()
)

## 3. Diagnosis-Level Claim Expansion

The standardized claim table contains up to ten ICD-9-CM diagnosis
positions per claim.

For HCC mapping, diagnoses must be analyzed individually.

This section expands each claim into diagnosis-level records while
preserving:

- claim identifier,
- beneficiary identifier,
- claim type,
- service date,
- diagnosis position,
- original ICD-9-CM diagnosis code.

The resulting diagnosis-level dataset forms the basis for ICD-9 → ICD-10
conversion and subsequent HCC V22/V28 mapping.

In [ ]:
df_diagnosis = (
    df_silver_claim
    .select(
        "claim_id",
        "member_id",
        "claim_type",
        "service_date",

        F.posexplode(
            "icd9_array"
        ).alias(
            "diagnosis_position",
            "icd9_code"
        )
    )
)

# posexplode() starts at zero, convert position into a human-readable 1–10 value
df_diagnosis = (
    df_diagnosis
    .withColumn(
        "diagnosis_position",
        F.col("diagnosis_position") + 1
    )
)

df_diagnosis = (
    df_diagnosis
    .withColumn(
        "icd9_code_normalized",
        F.upper(
            F.regexp_replace(
                F.trim("icd9_code"),
                "\\.",
                ""
            )
        )
    )
)

In [ ]:
display(
    df_diagnosis.limit(25)
)

In [ ]:
diagnosis_validation = (
    df_diagnosis
    .agg(
        F.count("*")
        .alias("diagnosis_rows"),

        F.countDistinct("claim_id")
        .alias("claims_with_diagnosis"),

        F.countDistinct("member_id")
        .alias("members_with_diagnosis"),

        F.countDistinct(
            "icd9_code_normalized"
        ).alias("distinct_icd9_codes"),

        F.sum(
            F.when(
                F.col("icd9_code_normalized").isNull(),
                1
            ).otherwise(0)
        ).alias("null_diagnoses")
    )
)

display(diagnosis_validation)

In [ ]:
diagnosis_position_dist = (
    df_diagnosis
    .groupBy(
        "claim_type",
        "diagnosis_position"
    )
    .count()
    .orderBy(
        "claim_type",
        "diagnosis_position"
    )
)

position_pd = diagnosis_position_dist.toPandas()

plt.figure(figsize=(9, 5))

for claim_type in position_pd["claim_type"].unique():

    subset = position_pd[
        position_pd["claim_type"] == claim_type
    ]

    plt.plot(
        subset["diagnosis_position"],
        subset["count"],
        marker="o",
        label=claim_type
    )

plt.title(
    "Diagnosis Position Coverage by Claim Type"
)

plt.xlabel(
    "Diagnosis position"
)

plt.ylabel(
    "Number of diagnosis records"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
df_diagnosis_mapped = (
    df_diagnosis.alias("d")
    .join(
        F.broadcast(
            df_gem_canonical
            .select(
                "icd9_code_normalized",
                "icd10_code_normalized",
                "approximate_flag",
                "combination_flag"
            )
            .alias("g")
        ),

        F.col("d.icd9_code_normalized")
        ==
        F.col("g.icd9_code_normalized"),

        "left"
    )

    .select(
        F.col("d.claim_id"),
        F.col("d.member_id"),
        F.col("d.claim_type"),
        F.col("d.service_date"),
        F.col("d.diagnosis_position"),
        F.col("d.icd9_code"),
        F.col("d.icd9_code_normalized"),

        F.col(
            "g.icd10_code_normalized"
        ).alias(
            "icd10_code"
        ),

        F.col(
            "g.approximate_flag"
        ),

        F.col(
            "g.combination_flag"
        )
    )
)

In [ ]:
mapping_coverage = (
    df_diagnosis_mapped
    .agg(
        F.count("*")
        .alias("diagnosis_rows"),

        F.sum(
            F.when(
                F.col("icd10_code").isNotNull(),
                1
            ).otherwise(0)
        ).alias("mapped_to_icd10"),

        F.sum(
            F.when(
                F.col("icd10_code").isNull(),
                1
            ).otherwise(0)
        ).alias("unmapped_icd9")
    )

    .withColumn(
        "mapping_pct",

        F.round(
            F.col("mapped_to_icd10")
            /
            F.col("diagnosis_rows")
            * 100,
            2
        )
    )
)

display(mapping_coverage)

In [ ]:
mapping_plot_df = (
    mapping_coverage
    .select(
        "mapped_to_icd10",
        "unmapped_icd9"
    )
    .toPandas()
)

labels = [
    "Mapped to ICD-10",
    "Unmapped ICD-9"
]

values = [
    mapping_plot_df.loc[
        0,
        "mapped_to_icd10"
    ],
    mapping_plot_df.loc[
        0,
        "unmapped_icd9"
    ]
]

plt.figure(figsize=(7, 5))

plt.bar(
    labels,
    values
)

plt.title(
    "ICD-9 → ICD-10 Diagnosis Mapping Coverage"
)

plt.ylabel(
    "Diagnosis records"
)

plt.tight_layout()
plt.show()

In [ ]:
df_v22_raw = spark.table(
    "Bronze_HCC.dbo.bronze_hcc_v22_mapping_raw"
)

df_v28_raw = spark.table(
    "Bronze_HCC.dbo.bronze_hcc_v28_mapping_raw"
)

In [ ]:
print("=== V22 Mapping Schema ===")
df_v22_raw.printSchema()

print("\n=== V28 Mapping Schema ===")
df_v28_raw.printSchema()

In [ ]:
display(df_v22_raw.limit(10))
display(df_v28_raw.limit(10))

## 4. CMS-HCC V22 and V28 Reference Standardization

The official CMS ICD-10 mapping files contain descriptive preamble and
multi-line header records before the diagnosis-level mapping rows.

Only valid ICD-10 diagnosis records are retained.

For this project:

- V22 HCC assignments are sourced from the 2017 CMS-HCC mapping.
- V28 HCC assignments are sourced from the 2024 CMS-HCC mapping.
- ICD-10 codes are normalized by removing periods and standardizing case.
- Duplicate diagnosis mappings are checked before downstream joins.

In [ ]:
df_v22_mapping = (
    df_v22_raw

    .select(
        F.upper(
            F.regexp_replace(
                F.trim(
                    F.col(
                        "icd_10_cm_codes_cms_hcc_and_rxhcc_models"
                    )
                ),
                "\\.",
                ""
            )
        ).alias("icd10_code"),

        F.trim(
            F.col("c1")
        ).alias("description"),

        F.col("c2")
        .cast("int")
        .alias("hcc_v22")
    )

    # Remove CMS title/header/preamble rows
    .filter(
        F.col("icd10_code").rlike(
            "^[A-Z][0-9][A-Z0-9]{1,5}$"
        )
    )

    # Keep diagnoses that actually map to V22
    .filter(
        F.col("hcc_v22").isNotNull()
    )

    .dropDuplicates(
        ["icd10_code"]
    )
)

In [ ]:
df_v28_mapping = (
    df_v28_raw

    .select(
        F.upper(
            F.regexp_replace(
                F.trim(
                    F.col(
                        "icd_10_cm_codes_esrd_cms_hcc_and_rxhcc_models"
                    )
                ),
                "\\.",
                ""
            )
        ).alias("icd10_code"),

        F.trim(
            F.col("c1")
        ).alias("description"),

        # Column G in official CMS file
        F.col("c6")
        .cast("int")
        .alias("hcc_v28"),

        # Column N = V28 payment-year indicator
        F.col("c13")
        .cast("string")
        .alias("v28_payment_year_flag")
    )

    .filter(
        F.col("icd10_code").rlike(
            "^[A-Z][0-9][A-Z0-9]{1,5}$"
        )
    )

    .filter(
        F.col("hcc_v28").isNotNull()
    )

    .dropDuplicates(
        ["icd10_code"]
    )
)

In [ ]:
mapping_reference_summary = [
    (
        "V22",
        df_v22_mapping.count(),
        df_v22_mapping
        .select("icd10_code")
        .distinct()
        .count(),
        df_v22_mapping
        .select("hcc_v22")
        .distinct()
        .count()
    ),

    (
        "V28",
        df_v28_mapping.count(),
        df_v28_mapping
        .select("icd10_code")
        .distinct()
        .count(),
        df_v28_mapping
        .select("hcc_v28")
        .distinct()
        .count()
    )
]

summary_df = spark.createDataFrame(
    mapping_reference_summary,
    [
        "model",
        "mapping_rows",
        "distinct_icd10_codes",
        "distinct_hcc_codes"
    ]
)

display(summary_df)

In [ ]:
mapping_counts_pd = (
    summary_df
    .select(
        "model",
        "mapping_rows"
    )
    .toPandas()
)

plt.figure(figsize=(7, 5))

plt.bar(
    mapping_counts_pd["model"],
    mapping_counts_pd["mapping_rows"]
)

plt.title(
    "ICD-10 Diagnosis Coverage: CMS-HCC V22 vs V28"
)

plt.xlabel("CMS-HCC model")
plt.ylabel("Mapped ICD-10 codes")

plt.tight_layout()
plt.show()

In [ ]:
(
    df_v22_mapping
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "silver_hcc_v22_mapping"
    )
)

(
    df_v28_mapping
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "silver_hcc_v28_mapping"
    )
)

In [ ]:
df_diagnosis_hcc = (
    df_diagnosis_mapped.alias("d")

    .join(
        F.broadcast(
            df_v22_mapping
            .select(
                "icd10_code",
                "hcc_v22"
            )
            .alias("v22")
        ),

        F.col("d.icd10_code")
        ==
        F.col("v22.icd10_code"),

        "left"
    )

    .drop(
        F.col("v22.icd10_code")
    )
)

In [ ]:
df_diagnosis_hcc = (
    df_diagnosis_hcc.alias("d")

    .join(
        F.broadcast(
            df_v28_mapping
            .select(
                "icd10_code",
                "hcc_v28"
            )
            .alias("v28")
        ),

        F.col("d.icd10_code")
        ==
        F.col("v28.icd10_code"),

        "left"
    )

    .drop(
        F.col("v28.icd10_code")
    )
)

In [ ]:
df_diagnosis_hcc = (
    df_diagnosis_hcc
    .withColumn(
        "hcc_mapping_status",

        F.when(
            F.col("hcc_v22").isNotNull()
            &
            F.col("hcc_v28").isNotNull(),
            "BOTH_V22_V28"
        )

        .when(
            F.col("hcc_v22").isNotNull(),
            "V22_ONLY"
        )

        .when(
            F.col("hcc_v28").isNotNull(),
            "V28_ONLY"
        )

        .otherwise(
            "NON_HCC_OR_UNMAPPED"
        )
    )
)

In [ ]:
hcc_status_summary = (
    df_diagnosis_hcc
    .groupBy(
        "hcc_mapping_status"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

display(hcc_status_summary)

In [ ]:
hcc_status_pd = (
    hcc_status_summary
    .toPandas()
)

plt.figure(figsize=(9, 5))

plt.bar(
    hcc_status_pd["hcc_mapping_status"],
    hcc_status_pd["count"]
)

plt.title(
    "Diagnosis-Level HCC Mapping Status: V22 vs V28"
)

plt.xlabel("Mapping status")
plt.ylabel("Diagnosis records")

plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
total_diagnoses = df_diagnosis_hcc.count()

hcc_status_pct = (
    hcc_status_summary
    .withColumn(
        "percentage",
        F.round(
            F.col("count")
            / F.lit(total_diagnoses)
            * 100,
            2
        )
    )
)

display(hcc_status_pct)

In [ ]:
display(
    df_diagnosis_hcc
    .groupBy("hcc_mapping_status")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count")
            / F.lit(df_diagnosis_hcc.count())
            * 100,
            2
        )
    )
    .orderBy(F.desc("count"))
)

In [ ]:
print("Before HCC mapping:", df_diagnosis_mapped.count())
print("After HCC mapping :", df_diagnosis_hcc.count())


In [ ]:
(
    df_diagnosis_hcc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_diagnosis_hcc")
)

## 6. Silver Member Population

Annual CMS beneficiary summary files are consolidated into a single
member-level analytical population.

For each beneficiary, the most recent available record from 2008–2010
is selected.

The Silver member table standardizes:

- member identifier,
- demographics,
- age and age band,
- beneficiary state,
- ESRD indicator,
- Medicare coverage months,
- and chronic-condition indicators.

This table becomes the demographic dimension used by downstream HCC,
RAF, cohort, ML, and Power BI analysis.

In [ ]:
# Combine the annual files:
df_bene_all = (
    df_bene_2008.withColumn("reference_year", F.lit(2008))
    .unionByName(
        df_bene_2009.withColumn("reference_year", F.lit(2009))
    )
    .unionByName(
        df_bene_2010.withColumn("reference_year", F.lit(2010))
    )
)

# Select the latest record:
member_window = (
    Window
    .partitionBy("DESYNPUF_ID")
    .orderBy(F.col("reference_year").desc())
)

df_member_latest = (
    df_bene_all
    .withColumn(
        "rn",
        F.row_number().over(member_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Create demographics:
df_silver_member = (
    df_member_latest

    .withColumn(
        "member_id",
        F.col("DESYNPUF_ID").cast("string")
    )

    .withColumn(
        "birth_date",
        F.to_date(
            F.col("BENE_BIRTH_DT").cast("string"),
            "yyyyMMdd"
        )
    )

    .withColumn(
        "reference_date",
        F.to_date(
            F.concat(
                F.col("reference_year").cast("string"),
                F.lit("1231")
            ),
            "yyyyMMdd"
        )
    )

    .withColumn(
        "age",
        F.floor(
            F.months_between(
                F.col("reference_date"),
                F.col("birth_date")
            ) / 12
        ).cast("int")
    )

    .withColumn(
        "gender",
        F.when(F.col("BENE_SEX_IDENT_CD") == 1, "Male")
        .when(F.col("BENE_SEX_IDENT_CD") == 2, "Female")
        .otherwise("Unknown")
    )

    .withColumn(
        "age_band",
        F.when(F.col("age") < 65, "<65")
        .when(F.col("age") <= 69, "65-69")
        .when(F.col("age") <= 74, "70-74")
        .when(F.col("age") <= 79, "75-79")
        .when(F.col("age") <= 84, "80-84")
        .otherwise("85+")
    )

    .withColumn(
        "has_diabetes",
        F.col("SP_DIABETES") == 1
    )

    .withColumn(
        "has_heart_failure",
        F.col("SP_CHF") == 1
    )

    .withColumn(
        "has_copd",
        F.col("SP_COPD") == 1
    )

    .withColumn(
        "has_cancer",
        F.col("SP_CNCR") == 1
    )

    .withColumn(
        "has_ckd",
        F.col("SP_CHRNKIDN") == 1
    )
)

In [ ]:
# Add chronic-condition count
df_silver_member = (
    df_silver_member
    .withColumn(
        "chronic_condition_count",
        sum(
            F.col(c).cast("int")
            for c in [
                "has_diabetes",
                "has_heart_failure",
                "has_copd",
                "has_cancer",
                "has_ckd"
            ]
        )
    )
)

print("Member rows:", df_silver_member.count())

print(
    "Distinct members:",
    df_silver_member
    .select("member_id")
    .distinct()
    .count()
)

In [ ]:
age_band_counts = (
    df_silver_member
    .groupBy("age_band")
    .count()
    .orderBy("age_band")
)

display(age_band_counts)

age_pd = age_band_counts.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(age_pd["age_band"], age_pd["count"])

plt.title("Beneficiary Population by Age Band")
plt.xlabel("Age band")
plt.ylabel("Members")

plt.tight_layout()
plt.show()

In [ ]:
(
    df_silver_member
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_member")
)

## 7. Member-Level V22 and V28 HCC Profiles

Diagnosis-level HCC assignments are aggregated to the beneficiary level.

For each CMS-HCC model, repeated occurrences of the same HCC across multiple
claims are counted only once per member.

The resulting member-level profile contains:

- unique V22 HCC assignments,
- unique V28 HCC assignments,
- V22 HCC count,
- V28 HCC count,
- HCC count difference between V28 and V22.

V22 and V28 HCC identifiers are treated as model-specific categories.
Matching numeric identifiers across models are not assumed to represent
equivalent clinical categories.

In [ ]:
# create unique member/HCC assignments
df_member_v22_unique = (
    df_diagnosis_hcc
    .filter(F.col("hcc_v22").isNotNull())
    .select(
        "member_id",
        F.col("hcc_v22").cast("int").alias("hcc_v22")
    )
    .dropDuplicates([
        "member_id",
        "hcc_v22"
    ])
)

df_member_v28_unique = (
    df_diagnosis_hcc
    .filter(F.col("hcc_v28").isNotNull())
    .select(
        "member_id",
        F.col("hcc_v28").cast("int").alias("hcc_v28")
    )
    .dropDuplicates([
        "member_id",
        "hcc_v28"
    ])
)

df_member_v22 = (
    df_member_v22_unique
    .groupBy("member_id")
    .agg(
        F.sort_array(
            F.collect_set("hcc_v22")
        ).alias("hcc_codes_v22"),

        F.countDistinct("hcc_v22")
        .alias("hcc_count_v22")
    )
)

df_member_v28 = (
    df_member_v28_unique
    .groupBy("member_id")
    .agg(
        F.sort_array(
            F.collect_set("hcc_v28")
        ).alias("hcc_codes_v28"),

        F.countDistinct("hcc_v28")
        .alias("hcc_count_v28")
    )
)

In [ ]:
df_member_hcc_profile = (
    df_silver_member
    .select(
        "member_id",
        "reference_year",
        "age",
        "age_band",
        "gender",
        "chronic_condition_count"
    )

    .join(
        df_member_v22,
        "member_id",
        "left"
    )

    .join(
        df_member_v28,
        "member_id",
        "left"
    )
)

member_hcc_validation = (
    df_member_hcc_profile
    .agg(
        F.count("*")
        .alias("member_rows"),

        F.countDistinct("member_id")
        .alias("distinct_members"),

        F.sum(
            F.when(
                F.col("hcc_count_v22") > 0,
                1
            ).otherwise(0)
        ).alias("members_with_v22_hcc"),

        F.sum(
            F.when(
                F.col("hcc_count_v28") > 0,
                1
            ).otherwise(0)
        ).alias("members_with_v28_hcc"),

        F.round(
            F.avg("hcc_count_v22"),
            2
        ).alias("avg_hcc_count_v22"),

        F.round(
            F.avg("hcc_count_v28"),
            2
        ).alias("avg_hcc_count_v28"),

        F.max("hcc_count_v22")
        .alias("max_hcc_count_v22"),

        F.max("hcc_count_v28")
        .alias("max_hcc_count_v28")
    )
)

display(member_hcc_validation)

In [ ]:
df_member_hcc_profile = (
    df_member_hcc_profile
    .withColumn(
        "hcc_count_delta_v28_v22",
        F.col("hcc_count_v28")
        - F.col("hcc_count_v22")
    )
    .withColumn(
        "hcc_count_direction",
        F.when(
            F.col("hcc_count_delta_v28_v22") > 0,
            "INCREASE"
        )
        .when(
            F.col("hcc_count_delta_v28_v22") < 0,
            "DECREASE"
        )
        .otherwise("NO_CHANGE")
    )
)

In [ ]:
hcc_direction_summary = (
    df_member_hcc_profile
    .groupBy("hcc_count_direction")
    .count()
    .orderBy(F.desc("count"))
)

display(hcc_direction_summary)

In [ ]:
direction_pd = hcc_direction_summary.toPandas()

plt.figure(figsize=(7, 5))

plt.bar(
    direction_pd["hcc_count_direction"],
    direction_pd["count"]
)

plt.title("Member HCC Count Change: V22 → V28")
plt.xlabel("HCC count direction")
plt.ylabel("Members")

plt.tight_layout()
plt.show()

In [ ]:
df_member_hcc_profile = (
    df_silver_member
    .select(
        "member_id",
        "reference_year",
        "age",
        "age_band",
        "gender",
        "chronic_condition_count"
    )
    .join(
        df_member_v22,
        "member_id",
        "left"
    )
    .join(
        df_member_v28,
        "member_id",
        "left"
    )
    .withColumn(
        "hcc_codes_v22",
        F.coalesce(
            F.col("hcc_codes_v22"),
            F.array().cast("array<int>")
        )
    )
    .withColumn(
        "hcc_codes_v28",
        F.coalesce(
            F.col("hcc_codes_v28"),
            F.array().cast("array<int>")
        )
    )
    .fillna({
        "hcc_count_v22": 0,
        "hcc_count_v28": 0
    })
    .withColumn(
        "hcc_count_delta_v28_v22",
        F.col("hcc_count_v28")
        - F.col("hcc_count_v22")
    )
    .withColumn(
        "hcc_count_direction",
        F.when(
            F.col("hcc_count_delta_v28_v22") > 0,
            "INCREASE"
        )
        .when(
            F.col("hcc_count_delta_v28_v22") < 0,
            "DECREASE"
        )
        .otherwise("NO_CHANGE")
    )
)

In [ ]:
display(
    df_member_hcc_profile
    .agg(
        F.count("*").alias("member_rows"),
        F.countDistinct("member_id").alias("distinct_members"),

        F.sum(
            F.when(F.col("hcc_count_v22") > 0, 1)
            .otherwise(0)
        ).alias("members_with_v22_hcc"),

        F.sum(
            F.when(F.col("hcc_count_v28") > 0, 1)
            .otherwise(0)
        ).alias("members_with_v28_hcc"),

        F.round(
            F.avg("hcc_count_v22"), 2
        ).alias("avg_hcc_count_v22"),

        F.round(
            F.avg("hcc_count_v28"), 2
        ).alias("avg_hcc_count_v28"),

        F.max("hcc_count_v22").alias("max_hcc_count_v22"),
        F.max("hcc_count_v28").alias("max_hcc_count_v28")
    )
)

In [ ]:
(
    df_member_hcc_profile
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_member_hcc_profile")
)

In [ ]:
print(
    "Member HCC profiles:",
    spark.table(
        "silver_member_hcc_profile"
    ).count()
)

In [ ]:
from hccpy.hcc import HCCEngine

engine_v22 = HCCEngine(
    version="22",
    cif=0,
    norm_params={"C": 1}
)

engine_v28 = HCCEngine(
    version="28",
    cif=0,
    norm_params={"C": 1}
)

print("V22 and V28 scoring engines loaded.")

In [ ]:
test_codes = [
    "E1169",
    "I5030",
    "I509"
]

test_v22 = engine_v22.profile(
    test_codes,
    age=70,
    sex="M",
    elig="CNA",
    orec="0",
    medicaid=False
)

test_v28 = engine_v28.profile(
    test_codes,
    age=70,
    sex="M",
    elig="CNA",
    orec="0",
    medicaid=False
)

print("V22 score:", test_v22["risk_score"])
print("V22 HCCs :", test_v22["hcc_lst"])

print()

print("V28 score:", test_v28["risk_score"])
print("V28 HCCs :", test_v28["hcc_lst"])

## 8. Standardized V22 and V28 Member Risk Scoring

Each beneficiary is scored under CMS-HCC V22 and V28 using the same standardized
community non-dual aged eligibility assumption.

The purpose is comparative model analysis rather than reproduction of an
individual beneficiary's official CMS payment RAF.

Normalization and coding-intensity adjustments are disabled so that score
differences primarily reflect changes in model structure, diagnosis mappings,
hierarchies, and interactions.

In [ ]:
df_member_icd10 = (
    df_diagnosis_mapped
    .filter(F.col("icd10_code").isNotNull())
    .groupBy("member_id")
    .agg(
        F.sort_array(
            F.collect_set("icd10_code")
        ).alias("icd10_codes")
    )
)

df_scoring_input = (
    df_silver_member
    .select(
        "member_id",
        "age",
        "gender"
    )
    .join(
        df_member_icd10,
        "member_id",
        "left"
    )
    .withColumn(
        "icd10_codes",
        F.coalesce(
            F.col("icd10_codes"),
            F.array().cast("array<string>")
        )
    )
)

print("Scoring population:", df_scoring_input.count())

In [ ]:
import pandas as pd
import re

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

scoring_schema = StructType([
    StructField("member_id", StringType(), False),
    StructField("risk_score_v22_std", DoubleType(), True),
    StructField("risk_score_v28_std", DoubleType(), True),
    StructField("payment_hcc_count_v22", IntegerType(), True),
    StructField("payment_hcc_count_v28", IntegerType(), True),
    StructField("scoring_status", StringType(), True)
])

In [ ]:
def score_members(iterator):

    from hccpy.hcc import HCCEngine
    import pandas as pd
    import re

    engine_v22 = HCCEngine(
        version="22",
        cif=0,
        norm_params={"C": 1}
    )

    engine_v28 = HCCEngine(
        version="28",
        cif=0,
        norm_params={"C": 1}
    )

    for pdf in iterator:

        output = []

        for row in pdf.itertuples(index=False):

            try:
                codes = (
                    list(row.icd10_codes)
                    if row.icd10_codes is not None
                    else []
                )

                age = int(row.age) if row.age is not None else 70

                sex = (
                    "M"
                    if row.gender == "Male"
                    else "F"
                )

                v22 = engine_v22.profile(
                    codes,
                    age=age,
                    sex=sex,
                    elig="CNA",
                    orec="0",
                    medicaid=False
                )

                v28 = engine_v28.profile(
                    codes,
                    age=age,
                    sex=sex,
                    elig="CNA",
                    orec="0",
                    medicaid=False
                )

                # Only true HCC terms: HCC85, HCC18, etc.
                # Excludes interactions and count variables.
                v22_hccs = [
                    x for x in v22["hcc_lst"]
                    if re.fullmatch(r"HCC\d+", x)
                ]

                v28_hccs = [
                    x for x in v28["hcc_lst"]
                    if re.fullmatch(r"HCC\d+", x)
                ]

                output.append({
                    "member_id": row.member_id,
                    "risk_score_v22_std":
                        float(v22["risk_score"]),

                    "risk_score_v28_std":
                        float(v28["risk_score"]),

                    "payment_hcc_count_v22":
                        len(v22_hccs),

                    "payment_hcc_count_v28":
                        len(v28_hccs),

                    "scoring_status": "SUCCESS"
                })

            except Exception as e:

                output.append({
                    "member_id": row.member_id,
                    "risk_score_v22_std": None,
                    "risk_score_v28_std": None,
                    "payment_hcc_count_v22": None,
                    "payment_hcc_count_v28": None,
                    "scoring_status":
                        "FAILED: " + str(e)[:150]
                })

        yield pd.DataFrame(output)



df_member_scores = (
    df_scoring_input
    .mapInPandas(
        score_members,
        schema=scoring_schema
    )
)

In [ ]:
(
    df_member_scores
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_member_risk_scores")
)

In [ ]:
df_member_scores = spark.table(
    "silver_member_risk_scores"
)

display(
    df_member_scores
    .groupBy("scoring_status")
    .count()
    .orderBy(F.desc("count"))
)

In [ ]:
display(
    df_member_scores
    .agg(
        F.count("*").alias("rows"),

        F.round(
            F.avg("risk_score_v22_std"), 3
        ).alias("avg_v22"),

        F.round(
            F.avg("risk_score_v28_std"), 3
        ).alias("avg_v28"),

        F.min("risk_score_v22_std").alias("min_v22"),
        F.max("risk_score_v22_std").alias("max_v22"),

        F.min("risk_score_v28_std").alias("min_v28"),
        F.max("risk_score_v28_std").alias("max_v28")
    )
)

In [ ]:
df_silver_final = (
    df_silver_member.alias("m")

    .join(
        df_member_hcc_profile
        .select(
            "member_id",
            "hcc_codes_v22",
            "hcc_codes_v28",
            "hcc_count_v22",
            "hcc_count_v28",
            "hcc_count_delta_v28_v22"
        )
        .alias("h"),
        "member_id",
        "left"
    )

    .join(
        df_member_scores
        .select(
            "member_id",
            "risk_score_v22_std",
            "risk_score_v28_std",
            "payment_hcc_count_v22",
            "payment_hcc_count_v28",
            "scoring_status"
        )
        .alias("s"),
        "member_id",
        "left"
    )

    .withColumn(
        "risk_score_delta",
        F.col("risk_score_v28_std")
        - F.col("risk_score_v22_std")
    )

    .withColumn(
        "risk_score_pct_change",
        F.when(
            F.col("risk_score_v22_std") > 0,
            (
                F.col("risk_score_delta")
                / F.col("risk_score_v22_std")
            ) * 100
        )
    )

    .withColumn(
        "risk_score_direction",
        F.when(
            F.col("risk_score_delta") > 0,
            "INCREASE"
        )
        .when(
            F.col("risk_score_delta") < 0,
            "DECREASE"
        )
        .otherwise("NO_CHANGE")
    )
)

In [ ]:
display(
    df_silver_final
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("member_id").alias("distinct_members"),

        F.round(
            F.avg("risk_score_v22_std"), 3
        ).alias("avg_v22"),

        F.round(
            F.avg("risk_score_v28_std"), 3
        ).alias("avg_v28"),

        F.round(
            F.avg("risk_score_delta"), 3
        ).alias("avg_delta")
    )
)

In [ ]:
display(
    df_silver_final
    .groupBy("risk_score_direction")
    .count()
    .orderBy(F.desc("count"))
)

(
    df_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_member_risk_final")
)